# 과제 2 — CDW 실무 사례 (20점)

**데이터베이스 관리 · 과제 2**

- **제출물**: 이 노트북 파일 하나 (모든 셀을 실행한 상태로 저장)
- **마감**: 13주차 수업 후 1주일, LMS 제출
- **배점**: 단계별 SQL 8점 + 검산표 4점 + 오류 발견·수정 설명 5점 + 결과 해석 3점 = 20점
- **범위**: 11~13주차 (CDW 업무 흐름 · 코호트 추출 · 오류 점검 · 검산)

### 이번 과제의 상황

여러분은 병원 의료정보팀의 분석 담당자입니다. 연구자에게서 아래 **분석 요청서**를 받았습니다.
요청대로 코호트를 추출하고, 중간에 **일부러 잘못 작성된 SQL 6개**를 찾아 고친 뒤,
검산표와 해석을 작성해 회신하는 것이 과제입니다.

## 0. 준비

In [ ]:
# ── 0. 준비: 이 셀을 먼저 실행하세요 (Shift+Enter) ───────────────────────────
import os, sys, urllib.request, hashlib
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand; break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "jupysql", "duckdb-engine"])

import duckdb, pandas as pd
pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
conn = duckdb.connect(DB, read_only=True)
%sql conn --alias aact

# ── 학번과 이름을 입력하세요 ────────────────────────────────────────────────
학번 = "20260000"     # ← 본인 학번으로 바꾸세요
이름 = "홍길동"        # ← 본인 이름으로 바꾸세요

MESH = ["Diabetes Mellitus, Type 2", "Hypertension", "Asthma", "Breast Neoplasms", "Depressive Disorder",
        "Stroke", "Heart Failure", "Pulmonary Disease, Chronic Obstructive", "Renal Insufficiency, Chronic",
        "Alzheimer Disease", "Obesity", "COVID-19", "Osteoarthritis", "Sepsis", "Pneumonia"]
MY_MESH = MESH[int(hashlib.md5(str(학번).encode()).hexdigest(), 16) % len(MESH)]
print(f"{이름}({학번}) 님의 배정 질환:  {MY_MESH}")
print("→ 아래 문항에서 '배정 질환' 이라고 나오면 이 값을 쓰세요.")

def _sig(df):
    d = df.copy()
    d.columns = [str(c).lower() for c in d.columns]
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(4)
    d = d.astype(str).sort_values(by=list(d.columns)).reset_index(drop=True)
    return hashlib.md5(d.to_csv(index=False).encode()).hexdigest()[:12]

def check(name, df):
    """직전 %%sql 결과를 _ 로 넘겨 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print("알 수 없는 문항:", name); return
    a = ANSWERS[name]
    exp = a["by_mesh"].get(MY_MESH) if a.get("by_mesh") else a
    if not isinstance(df, pd.DataFrame):
        print(f"❌ {name}  결과가 표가 아닙니다. %%sql 셀을 먼저 실행한 뒤 바로 다음 셀에서 check 하세요."); return
    if _sig(df) == exp["sig"]:
        print(f"✅ {name} 정답!  (행 {len(df)}개)")
    else:
        msg = f"❌ {name} 아직 다릅니다.  기대 행 수 {exp['rows']}개, 현재 {len(df)}개."
        if a.get("hint"): msg += "  힌트: " + a["hint"]
        print(msg)

def 제출확인():
    done = [k for k in ANSWERS if k.startswith("Q")]
    print(f"문항 {len(done)}개.  모든 셀을 실행한 뒤 파일을 저장해 제출하세요.")
    print(f"제출자: {이름} ({학번}) / 배정 질환: {MY_MESH}")

In [ ]:
ANSWERS = {
 "S1": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "a36720ec82a4",
    "rows": 1
   },
   "Hypertension": {
    "sig": "65bc33611f70",
    "rows": 1
   },
   "Asthma": {
    "sig": "3c217c64637d",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "a5417d7ecd81",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "34a03fba1cd3",
    "rows": 1
   },
   "Stroke": {
    "sig": "b5a82ccd38f2",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "7debba04f4ef",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "3c11b100444f",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "34a03fba1cd3",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "76ce11e1ebfd",
    "rows": 1
   },
   "Obesity": {
    "sig": "56f045cb4989",
    "rows": 1
   },
   "COVID-19": {
    "sig": "ab74fa505ac7",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "6cbd68518849",
    "rows": 1
   },
   "Sepsis": {
    "sig": "8a07147d52f1",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "e7c2816c9704",
    "rows": 1
   }
  },
  "hint": "COUNT(DISTINCT nct_id)"
 },
 "S2": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "ce05fb14948e",
    "rows": 1
   },
   "Hypertension": {
    "sig": "208723124f2c",
    "rows": 1
   },
   "Asthma": {
    "sig": "ae1ee0af2657",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "5e0859b90d82",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "fc8fa4ae1a4b",
    "rows": 1
   },
   "Stroke": {
    "sig": "f5c7660ec7c3",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "fc92e6f6f843",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "99b38c5a360c",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "59b3e0fa6004",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "716ff73db70c",
    "rows": 1
   },
   "Obesity": {
    "sig": "364bd3ff00a3",
    "rows": 1
   },
   "COVID-19": {
    "sig": "e23b11704fa5",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "78f4d1484ae2",
    "rows": 1
   },
   "Sepsis": {
    "sig": "d61bd8c95f5b",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "b860a2c74256",
    "rows": 1
   }
  },
  "hint": "<> 'WITHDRAWN'"
 },
 "S3": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "f5107ada5007",
    "rows": 1
   },
   "Hypertension": {
    "sig": "ce3b41300b23",
    "rows": 1
   },
   "Asthma": {
    "sig": "9cd14e0ae001",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "9a23fda673a6",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "c8a813d0039f",
    "rows": 1
   },
   "Stroke": {
    "sig": "8d6227b13e6a",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "a66991066718",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "5b74d536b2dd",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "ff6c77910ebe",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "1b3150913192",
    "rows": 1
   },
   "Obesity": {
    "sig": "05263ad5eeba",
    "rows": 1
   },
   "COVID-19": {
    "sig": "38c835d37301",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "9f172b1d3280",
    "rows": 1
   },
   "Sepsis": {
    "sig": "8c69506c415c",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "74392bde0dc2",
    "rows": 1
   }
  },
  "hint": "분모와 분자를 한 번에"
 },
 "S4": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "2120624e039b",
    "rows": 5
   },
   "Hypertension": {
    "sig": "16497d31db56",
    "rows": 5
   },
   "Asthma": {
    "sig": "a53e149684fb",
    "rows": 5
   },
   "Breast Neoplasms": {
    "sig": "0757ea8b96dd",
    "rows": 5
   },
   "Depressive Disorder": {
    "sig": "95cf6683985d",
    "rows": 5
   },
   "Stroke": {
    "sig": "49c0e48850e5",
    "rows": 5
   },
   "Heart Failure": {
    "sig": "2f2fa38365bc",
    "rows": 5
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "7be3f12978db",
    "rows": 5
   },
   "Renal Insufficiency, Chronic": {
    "sig": "847a49008593",
    "rows": 5
   },
   "Alzheimer Disease": {
    "sig": "1d26c26ee112",
    "rows": 5
   },
   "Obesity": {
    "sig": "82cc1cdfb54c",
    "rows": 5
   },
   "COVID-19": {
    "sig": "7ba1dac329f1",
    "rows": 5
   },
   "Osteoarthritis": {
    "sig": "983a180c991d",
    "rows": 5
   },
   "Sepsis": {
    "sig": "ba8435a37eb1",
    "rows": 5
   },
   "Pneumonia": {
    "sig": "07a5ba14ee3f",
    "rows": 5
   }
  },
  "hint": "removed = false 조건과 LIMIT 5"
 },
 "E1": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "a68de26d172d",
    "rows": 1
   },
   "Hypertension": {
    "sig": "8d40bdbed160",
    "rows": 1
   },
   "Asthma": {
    "sig": "4e98174d9ad4",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "fee8f9ab0e90",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "eaa4444eb026",
    "rows": 1
   },
   "Stroke": {
    "sig": "868cb7492998",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "9fb3dbd0b7d2",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "6c9e253d255c",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "eaa4444eb026",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "360740c4d0eb",
    "rows": 1
   },
   "Obesity": {
    "sig": "9f000883a861",
    "rows": 1
   },
   "COVID-19": {
    "sig": "34cb701c56b5",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "0cad7f8fbe9f",
    "rows": 1
   },
   "Sepsis": {
    "sig": "1aa0f4732514",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "3cb340bf989b",
    "rows": 1
   }
  },
  "hint": "JOIN 뒤에는 COUNT(DISTINCT 키)"
 },
 "E2": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "890cd8032d34",
    "rows": 1
   },
   "Hypertension": {
    "sig": "4e6816b302b7",
    "rows": 1
   },
   "Asthma": {
    "sig": "f5365f620c03",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "2fdae6b77b36",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "5bbfacd3e3c7",
    "rows": 1
   },
   "Stroke": {
    "sig": "1ec0c55c0bdf",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "9fb04b48bba2",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "cc1304807761",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "5bbfacd3e3c7",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "6fa66470942c",
    "rows": 1
   },
   "Obesity": {
    "sig": "0387def22e9c",
    "rows": 1
   },
   "COVID-19": {
    "sig": "23c08d30f766",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "dd403283cd86",
    "rows": 1
   },
   "Sepsis": {
    "sig": "22b9c9ca9989",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "8e8f3d02e05e",
    "rows": 1
   }
  },
  "hint": "INNER JOIN 을 LEFT JOIN 으로"
 },
 "E3": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "a8e1e97def30",
    "rows": 1
   },
   "Hypertension": {
    "sig": "327500b79864",
    "rows": 1
   },
   "Asthma": {
    "sig": "71045cd011a1",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "412ef73ab324",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "8549f0f9a772",
    "rows": 1
   },
   "Stroke": {
    "sig": "b75650871720",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "e95fa8a97f89",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "efb8ef794280",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "3b4138580cc4",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "f953a296e025",
    "rows": 1
   },
   "Obesity": {
    "sig": "3f8c8738bbf5",
    "rows": 1
   },
   "COVID-19": {
    "sig": "34d90ab6aa06",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "e4c7fbaa74d9",
    "rows": 1
   },
   "Sepsis": {
    "sig": "41256a973f7a",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "5c40d77a58e8",
    "rows": 1
   }
  },
  "hint": "= NULL 이 아니라 IS NULL"
 },
 "E4": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "0f512bd8a224",
    "rows": 1
   },
   "Hypertension": {
    "sig": "bc24aec7446b",
    "rows": 1
   },
   "Asthma": {
    "sig": "3ab227c98679",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "da701c48b8fa",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "a7d9eaac48b2",
    "rows": 1
   },
   "Stroke": {
    "sig": "de4ccc1d7ec4",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "10a9a0ad899e",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "98bd740717e0",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "bd92e3ab34a3",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "cb76dfce0c32",
    "rows": 1
   },
   "Obesity": {
    "sig": "068239b4b69c",
    "rows": 1
   },
   "COVID-19": {
    "sig": "689b1968ee74",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "f6d11ffe67c2",
    "rows": 1
   },
   "Sepsis": {
    "sig": "dfdde5105472",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "ed9283094b92",
    "rows": 1
   }
  },
  "hint": "removed = false 조건을 빠뜨리지 않기"
 },
 "E5": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "15f8d913d8c8",
    "rows": 1
   },
   "Hypertension": {
    "sig": "343cd5b31cf1",
    "rows": 1
   },
   "Asthma": {
    "sig": "c5063d00e62c",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "7755a9efbd98",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "c1e13409a508",
    "rows": 1
   },
   "Stroke": {
    "sig": "175b58d69c16",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "752edefb793a",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "1148f16e5888",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "eb6a97f274b5",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "011e7877eb40",
    "rows": 1
   },
   "Obesity": {
    "sig": "ce9b3b1cbe7b",
    "rows": 1
   },
   "COVID-19": {
    "sig": "1705803e082d",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "731e61e490af",
    "rows": 1
   },
   "Sepsis": {
    "sig": "8dc5d4078b76",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "98bf0bc2ad6c",
    "rows": 1
   }
  },
  "hint": "실제 값은 대문자 COMPLETED"
 },
 "E6": {
  "by_mesh": {
   "Diabetes Mellitus, Type 2": {
    "sig": "601346743e4a",
    "rows": 1
   },
   "Hypertension": {
    "sig": "d77a3a9f8c4a",
    "rows": 1
   },
   "Asthma": {
    "sig": "676c46cd0bf3",
    "rows": 1
   },
   "Breast Neoplasms": {
    "sig": "46e17107fe7f",
    "rows": 1
   },
   "Depressive Disorder": {
    "sig": "f80fe3925d01",
    "rows": 1
   },
   "Stroke": {
    "sig": "961dba42eb50",
    "rows": 1
   },
   "Heart Failure": {
    "sig": "eb1295ee862b",
    "rows": 1
   },
   "Pulmonary Disease, Chronic Obstructive": {
    "sig": "f567a9f568e4",
    "rows": 1
   },
   "Renal Insufficiency, Chronic": {
    "sig": "f567a9f568e4",
    "rows": 1
   },
   "Alzheimer Disease": {
    "sig": "0bf28794554f",
    "rows": 1
   },
   "Obesity": {
    "sig": "e48bc5887075",
    "rows": 1
   },
   "COVID-19": {
    "sig": "a60e0e8bf4e2",
    "rows": 1
   },
   "Osteoarthritis": {
    "sig": "c97cc94f1872",
    "rows": 1
   },
   "Sepsis": {
    "sig": "0d6d507d55d9",
    "rows": 1
   },
   "Pneumonia": {
    "sig": "0520fbf31e17",
    "rows": 1
   }
  },
  "hint": "100.0 을 곱해서 소수 계산"
 }
}

## 1. 분석 요청서

| 항목 | 내용 |
|---|---|
| 연구 질문 | **배정 질환**의 임상시험은 어떤 상황이며, 결과 게시율은 얼마인가? |
| 대상 (포함) | 2015~2024년에 시작한 시험 중 MeSH 용어가 **배정 질환**인 것 |
| 제외 | 진행 상태가 `WITHDRAWN`(철회)인 시험 |
| 분모 | 위 중 `COMPLETED`(완료)된 시험 |
| 분자 | 분모 중 결과 게시일(`results_first_posted_date`)이 있는 시험 |
| 필요 변수 | 등록번호, 시작일, 진행 상태, 결과게시일, 국가, 스폰서 유형 |
| 결과 형태 | 단계별 건수표(검산표) + 국가별 표 + 해석 1쪽 |

> **주의**: 한 시험에 질환이 여러 개 붙을 수 있습니다. 세는 단위는 **시험 1건**입니다.

## 2. 단계별 코호트 추출 (8점)

요청서의 조건을 **한 번에 하나씩** 적용하고, 각 단계의 건수를 4장의 검산표에 적습니다.

**1단계.** 배정 질환을 대상으로 한 시험이 몇 건인지 `대상시험` 으로 세세요. (`browse_conditions`)

In [ ]:
%%sql
-- S1
SELECT COUNT(DISTINCT nct_id) AS 대상시험
FROM browse_conditions
WHERE mesh_term = ___

In [ ]:
check("S1", _)

**2단계.** 위에서 철회(`WITHDRAWN`)된 시험을 뺀 건수를 `철회제외후` 로 세세요. (studies 와 JOIN)

In [ ]:
%%sql
-- S2
SELECT COUNT(DISTINCT s.nct_id) AS 철회제외후
FROM studies AS s
JOIN browse_conditions AS b ON ___
WHERE b.mesh_term = ___
  AND s.overall_status ___

In [ ]:
check("S2", _)

**3단계.** 완료된 시험 수(`분모`)와 그중 결과가 게시된 시험 수(`분자`)를 한 번에 구하세요.

In [ ]:
%%sql
-- S3
SELECT COUNT(DISTINCT s.nct_id) AS 분모,
       COUNT(DISTINCT CASE WHEN ___ THEN s.nct_id END) AS 분자
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = ___
  AND s.overall_status = ___

In [ ]:
check("S3", _)

**4단계.** 국가(`국가`)별 시험 수(`시험수`)를 많은 순으로 상위 5개만 조회하세요. 제외된 국가 행은 빼세요.

In [ ]:
%%sql
-- S4
SELECT co.name AS 국가, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN countries AS co ON ___
WHERE b.mesh_term = ___
  AND ___
GROUP BY co.name
ORDER BY 시험수 DESC
LIMIT 5

In [ ]:
check("S4", _)

## 3. 잘못 작성된 SQL 고치기 (5점)

아래 여섯 개의 SQL은 **동료가 작성한 것인데 모두 잘못되어 있습니다.**
12주차에서 배운 오류 유형을 떠올리며 고치세요. **대부분 오류 메시지가 나오지 않습니다.**

각 문항 아래의 마크다운 칸에 **무엇이 왜 틀렸는지** 한두 문장으로 적으면 설명 점수가 반영됩니다.

**E1.** 동료의 SQL — 배정 질환 시험 수를 세려고 했습니다.

아래 셀의 SQL을 **고쳐서** 실행하세요. (`배정질환을_여기에` 도 본인 질환으로 바꾸세요)

In [ ]:
%%sql
-- E1
SELECT COUNT(*) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = '배정질환을_여기에'

In [ ]:
check("E1", _)

> **E1 무엇이 왜 틀렸나요?**  (여기를 두 번 눌러 작성)
>
> 답:

**E2.** 동료의 SQL — “전체 대상 시험”을 분모로 세려고 했습니다. (기관 정보가 없는 시험도 분모에 넣어야 합니다)

아래 셀의 SQL을 **고쳐서** 실행하세요. (`배정질환을_여기에` 도 본인 질환으로 바꾸세요)

In [ ]:
%%sql
-- E2
SELECT COUNT(DISTINCT s.nct_id) AS 전체분모
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN facilities AS f ON s.nct_id = f.nct_id
WHERE b.mesh_term = '배정질환을_여기에'

In [ ]:
check("E2", _)

> **E2 무엇이 왜 틀렸나요?**  (여기를 두 번 눌러 작성)
>
> 답:

**E3.** 동료의 SQL — 결과가 게시되지 않은 시험을 세려고 했습니다.

아래 셀의 SQL을 **고쳐서** 실행하세요. (`배정질환을_여기에` 도 본인 질환으로 바꾸세요)

In [ ]:
%%sql
-- E3
SELECT COUNT(DISTINCT s.nct_id) AS 결과없음
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = '배정질환을_여기에'
  AND s.results_first_posted_date = NULL

In [ ]:
check("E3", _)

> **E3 무엇이 왜 틀렸나요?**  (여기를 두 번 눌러 작성)
>
> 답:

**E4.** 동료의 SQL — 미국에서 진행된 시험을 세려고 했습니다.

아래 셀의 SQL을 **고쳐서** 실행하세요. (`배정질환을_여기에` 도 본인 질환으로 바꾸세요)

In [ ]:
%%sql
-- E4
SELECT COUNT(DISTINCT s.nct_id) AS 미국시험
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN countries AS co ON s.nct_id = co.nct_id
WHERE b.mesh_term = '배정질환을_여기에'
  AND co.name = 'United States'

In [ ]:
check("E4", _)

> **E4 무엇이 왜 틀렸나요?**  (여기를 두 번 눌러 작성)
>
> 답:

**E5.** 동료의 SQL — 완료된 시험을 세려고 했습니다.

아래 셀의 SQL을 **고쳐서** 실행하세요. (`배정질환을_여기에` 도 본인 질환으로 바꾸세요)

In [ ]:
%%sql
-- E5
SELECT COUNT(DISTINCT s.nct_id) AS 완료시험
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = '배정질환을_여기에'
  AND s.overall_status = 'Completed'

In [ ]:
check("E5", _)

> **E5 무엇이 왜 틀렸나요?**  (여기를 두 번 눌러 작성)
>
> 답:

**E6.** 동료의 SQL — 완료된 시험의 결과 게시율(%)을 구하려고 했습니다. (소수 첫째 자리까지)

아래 셀의 SQL을 **고쳐서** 실행하세요. (`배정질환을_여기에` 도 본인 질환으로 바꾸세요)

In [ ]:
%%sql
-- E6
SELECT ROUND(100 *
  COUNT(DISTINCT CASE WHEN s.results_first_posted_date IS NOT NULL THEN s.nct_id END)
  / COUNT(DISTINCT s.nct_id), 1) AS 게시율
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = '배정질환을_여기에'
  AND s.overall_status = 'COMPLETED'

In [ ]:
check("E6", _)

> **E6 무엇이 왜 틀렸나요?**  (여기를 두 번 눌러 작성)
>
> 답:

## 4. 검산표 (4점)

위에서 구한 숫자를 아래 표에 채우세요. (표를 두 번 눌러 편집)

| 단계 | 적용한 조건 | 건수 | 앞 단계 대비 변화 |
|---|---|---|---|
| ① 전체 시험 | 조건 없음 | 37,509 | — |
| ② 배정 질환 대상 | mesh_term 조건 | | |
| ③ 철회 제외 | WITHDRAWN 제외 | | |
| ④ 완료된 것만 (분모) | COMPLETED | | |
| ⑤ 결과 게시 (분자) | 결과게시일 있음 | | |
| ⑥ 게시율 | ⑤ / ④ | % | — |

**추가로 확인한 것** (표를 채우며 알게 된 점을 적으세요)

- 기관 정보가 없어 INNER JOIN에서 빠지는 시험 수: _____건 (E2에서 확인)
- 제외된 국가 행(`removed = true`) 때문에 달라지는 미국 시험 수: _____건 (E4에서 확인)

## 5. 결과 해석 (3점)

아래 세 가지를 각각 문장으로 작성하세요. (전체 10~15문장)

**(1) 말할 수 있는 것.** 이 분석으로 확인된 사실을 조건과 함께 적으세요.
(배정 질환, 기간, 분모·분자의 정의, 게시율)

> 답:

**(2) 말할 수 없는 것.** 이 결과로 단정할 수 없는 것을 두 가지 이상 적고 이유를 쓰세요.

> 답:

**(3) 자료의 한계.** 사용한 데이터(aact_mini)의 한계를 적으세요.
(표본 비율, 자료 기준일, 등록된 시험만 포함된다는 점 등)

> 답:

## 6. 제출 전 확인

In [ ]:
제출확인()

제출 전 체크리스트

- [ ] 준비 셀에 **학번과 이름**을 입력했다
- [ ] 1~4단계(S1~S4)가 모두 ✅ 이다
- [ ] 오류 6개(E1~E6)를 모두 고쳐 ✅ 이고, **무엇이 왜 틀렸는지** 적었다
- [ ] 검산표를 채웠다
- [ ] 해석 (1)(2)(3)을 작성했다
- [ ] **모든 셀을 위에서부터 실행**한 상태로 저장했다
- [ ] 파일 이름을 `과제2_학번_이름.ipynb` 로 바꾸었다